# 04 - Arrhythmia Feasibility Analysis & Multi-Class ECG Modeling
## Multi-Class Electrophysiological Categorization (UCI Arrhythmia Dataset)

> **CRITICAL ARCHITECTURAL & CLINICAL FEASIBILITY DIRECTIVES (Rule 7, 10 & 20)**:
> 1. **Input Modality**: The dataset consists of **279 quantitative numerical features derived from 12-lead ECG waveforms** (intervals, vector angles, wave deflections across leads DI-V6).
> 2. **Symptom Infeasibility**: This model **CANNOT** predict arrhythmia from patient symptoms (palpitations, dizziness, syncope) or routine laboratory tests.
> 3. **Signal/Image Infeasibility**: Precomputed numerical features **do NOT prove** that the application can interpret scanned PDF reports, paper ECGs, or raw time-series voltage waveforms.
> 4. **Rare Class Limitation**: `Structural_Atrial_Abnormality` has only 9 instances in the entire cohort, resulting in severe statistical uncertainty for that subgroup.


In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

PROJECT_ROOT = Path('.').resolve().parent.parent if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from Machine_Learning.src.data_loader import load_arrhythmia_data
from Machine_Learning.src.preprocessing import prepare_arrhythmia_splits
from Machine_Learning.src.train_arrhythmia import (
    get_arrhythmia_candidate_models,
    train_and_select_best_arrhythmia_model,
)
from Machine_Learning.src.evaluate import evaluate_multiclass_classifier, print_multiclass_summary

sns.set_theme(style='whitegrid', palette='muted')
print('Environment initialized successfully. Project root:', PROJECT_ROOT)


Environment initialized successfully. Project root: C:\Users\lodha\Downloads\Heart_Disease_Screening


## 1. Feature Representation & Clinical Feasibility Assessment
Inspection of the 280 columns confirms:
- Columns 0-3: Demographics (Age, Sex, Height, Weight).
- Columns 4-14: Global intervals (QRS, PR, QT, T, P) and vector angles. Column 13 (J-angle) has 83.3% missingness and is dropped.
- Columns 15-278: 264 waveform deflection metrics across 12 standard leads.
**Verdict**: A patient-facing symptom intake form cannot supply these 279 parameters. Outpatient screening triage must mark arrhythmia ML prediction as **UNSUPPORTED WITHOUT 12-LEAD ECG**.


In [2]:
df_raw = load_arrhythmia_data()
splits = prepare_arrhythmia_splits(
    df_raw, test_size=0.2, val_size=0.1, random_state=42
)

classes = sorted(list(np.unique(splits.y_train)))
print(f'Total Dataset Records: {len(df_raw)}')
print(f'Training fold       : {splits.X_train.shape[0]} samples ({round(splits.X_train.shape[0]/len(df_raw)*100, 1)}%)')
print(f'Validation fold     : {splits.X_val.shape[0]} samples ({round(splits.X_val.shape[0]/len(df_raw)*100, 1)}%)')
print(f'Held-out Test fold  : {splits.X_test.shape[0]} samples ({round(splits.X_test.shape[0]/len(df_raw)*100, 1)}%)')
print(f'Transformed Features: {splits.X_train.shape[1]}')
print(f'Classes ({len(classes)}): {classes}')


2026-10-15 00:14:03,178 - Machine_Learning.src.data_loader - INFO - Loaded Arrhythmia dataset: shape=(430, 280) from C:\Users\lodha\Downloads\Heart_Disease_Screening\Machine_Learning\datasets\raw\arrhythmia_conditions.csv


Total Dataset Records: 430
Training fold       : 301 samples (70.0%)
Validation fold     : 43 samples (10.0%)
Held-out Test fold  : 86 samples (20.0%)
Transformed Features: 284
Classes (5): [np.str_('Conduction_Abnormality'), np.str_('Ischemic_MI_Related'), np.str_('Normal'), np.str_('Rhythm_Abnormality'), np.str_('Structural_Atrial_Abnormality')]


## 2. Multi-Class Candidate Model Comparison
Comparing candidate multi-class classifiers on the validation fold ($n=43$) using **Balanced Accuracy (Macro Recall)** as the primary selection criterion:
1. **Dummy Baseline**: Stratified chance classifier
2. **Logistic Regression (Balanced)**: Multinomial regularized linear classifier
3. **Random Forest (Depth 6 Balanced)**: Ensemble of 150 balanced trees
4. **Gradient Boosting**: Sequential boosting model
5. **Support Vector Classifier (RBF Balanced)**: Non-linear kernel


In [3]:
best_name, best_model, val_results = train_and_select_best_arrhythmia_model(splits, random_state=42)

val_summary = []
for name, res in val_results.items():
    val_summary.append({
        'Model': name,
        'Val Accuracy': res['accuracy'],
        'Val Balanced Acc (Macro Recall)': res['balanced_accuracy'],
        'Val Macro Prec': res['macro_precision'],
        'Val Macro F1': res['macro_f1'],
        'Val Weighted F1': res['weighted_f1'],
        'Val Macro ROC-AUC': res.get('macro_roc_auc_ovr')
    })

val_df = pd.DataFrame(val_summary).sort_values(by='Val Balanced Acc (Macro Recall)', ascending=False)
print('Arrhythmia Candidate Model Validation Comparison:')
print(val_df.to_string(index=False))


2026-10-15 00:14:03,462 - Machine_Learning.src.train_arrhythmia - INFO - Evaluating 5 candidate models on Arrhythmia validation fold across 5 classes...


2026-10-15 00:14:03,514 - Machine_Learning.src.train_arrhythmia - INFO - Candidate [Dummy (Stratified Baseline)]: Val Balanced Acc=0.1906, Val Macro F1=0.1809, Val Accuracy=0.3721


2026-10-15 00:14:03,650 - Machine_Learning.src.train_arrhythmia - INFO - Candidate [Logistic Regression (Balanced)]: Val Balanced Acc=0.5078, Val Macro F1=0.4899, Val Accuracy=0.7442


2026-10-15 00:14:04,391 - Machine_Learning.src.train_arrhythmia - INFO - Candidate [Random Forest (Depth 6 Balanced)]: Val Balanced Acc=0.5935, Val Macro F1=0.5840, Val Accuracy=0.8140


2026-10-15 00:14:10,568 - Machine_Learning.src.train_arrhythmia - INFO - Candidate [Gradient Boosting (Multiclass)]: Val Balanced Acc=0.4792, Val Macro F1=0.4503, Val Accuracy=0.7209


C:\Python314\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
2026-10-15 00:14:10,746 - Machine_Learning.src.train_arrhythmia - INFO - Candidate [Support Vector Classifier (RBF Balanced)]: Val Balanced Acc=0.3570, Val Macro F1=0.3329, Val Accuracy=0.6512


2026-10-15 00:14:10,748 - Machine_Learning.src.train_arrhythmia - INFO - Selected top Arrhythmia model: 'Random Forest (Depth 6 Balanced)' (Validation Balanced Accuracy: 0.5935)


Arrhythmia Candidate Model Validation Comparison:
                                   Model  Val Accuracy  Val Balanced Acc (Macro Recall)  Val Macro Prec  Val Macro F1  Val Weighted F1  Val Macro ROC-AUC
        Random Forest (Depth 6 Balanced)        0.8140                           0.5935          0.5852        0.5840           0.8040             0.8848
          Logistic Regression (Balanced)        0.7442                           0.5078          0.4819        0.4899           0.7248             0.9395
          Gradient Boosting (Multiclass)        0.7209                           0.4792          0.4481        0.4503           0.6864             0.7759
Support Vector Classifier (RBF Balanced)        0.6512                           0.3570          0.3227        0.3329           0.6309             0.9159
             Dummy (Stratified Baseline)        0.3721                           0.1906          0.1727        0.1809           0.3616             0.4884


## 3. Final Evaluation on Held-Out Test Data
Evaluating the top-performing model (`Random Forest (Depth 6 Balanced)`) on the unseen Test fold ($n=86$).


In [4]:
y_test_pred = best_model.predict(splits.X_test)
y_test_prob = best_model.predict_proba(splits.X_test)

test_metrics = evaluate_multiclass_classifier(splits.y_test, y_test_pred, y_test_prob, class_labels=classes)
print_multiclass_summary(test_metrics, f'HELD-OUT TEST SET ({best_name})')



MULTICLASS EVALUATION SUMMARY: HELD-OUT TEST SET (Random Forest (Depth 6 Balanced))
Sample Count       : 86
Overall Accuracy   : 0.6744
Balanced Accuracy  : 0.4884
Macro Precision    : 0.5055
Macro Recall       : 0.4884
Macro F1 Score     : 0.4934
Weighted F1 Score  : 0.6691
Macro ROC-AUC (OvR): 0.9016

Per-Class Breakdown:
  Conduction_Abnormality        : Recall=0.5000, Prec=0.5455, F1=0.5217 (N=12)
  Ischemic_MI_Related           : Recall=0.6667, Prec=0.5556, F1=0.6061 (N=15)
  Normal                        : Recall=0.7755, Prec=0.7600, F1=0.7677 (N=49)
  Rhythm_Abnormality            : Recall=0.5000, Prec=0.6667, F1=0.5714 (N=8)
  Structural_Atrial_Abnormality : Recall=0.0000, Prec=0.0000, F1=0.0000 (N=2)



## 4. Multi-Class Confusion Matrix Heatmap
Visualizing classification performance across condition groups.


In [5]:
cm = confusion_matrix(splits.y_test, y_test_pred, labels=classes)
cm_df = pd.DataFrame(cm, index=classes, columns=classes)

plt.figure(figsize=(9, 7))
sns.heatmap(cm_df, annot=True, fmt='d', cmap='Purples', cbar=False)
plt.title(f'Arrhythmia Confusion Matrix\n({best_name})')
plt.xlabel('Predicted Condition Group')
plt.ylabel('Actual Condition Group')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'Machine_Learning' / 'notebooks' / 'arrhythmia_confusion_matrix.png', dpi=150)
plt.close()
print('Saved diagnostic plot: arrhythmia_confusion_matrix.png')


Saved diagnostic plot: arrhythmia_confusion_matrix.png


## 5. Feasibility Verdict & Clinical Scope Declarations
1. **Supported Use**: Multi-class categorization of precomputed, digitized 12-lead ECG waveform measurements.
2. **Explicit Unsupported Declarations (Rule 7 & 20)**:
   - **Symptom-Based Screening**: **UNSUPPORTED**. Patient questionnaires (palpitations, dizziness) cannot substitute for ECG waveform deflections.
   - **Raw Signal Interpretation**: **UNSUPPORTED**. Precomputed tabular features do not enable direct parsing of raw voltage signals or scanned PDF waveforms.
3. **Rare Class Performance**:
   - `Structural_Atrial_Abnormality` ($n=2$ in test set) achieved $0\%$ recall despite class weighting. This demonstrates the hard statistical ceiling of extreme rarity ($n=9$ in entire cohort).
4. **Clinical Policy**: When patients present with palpitations in outpatient intake, the application must **NOT** guess arrhythmia risk. It must output an explicit `PENDING_ECG` status and recommend an ambulatory 12-lead ECG or Holter monitor.
